In [ ]:
import json
from datetime import datetime
import pycountry
from matplotlib import pyplot as plt
import pycountry_convert as pc

from emu_renewal.inputs import DATA_PATH, get_dec_pooled_totals, get_continent_data, get_continent_vars, get_country_vars, extract_specific_var
from emu_renewal.run import find_run_end_time, get_country_vacc_data

In [ ]:
countries = json.load(open(DATA_PATH / f"config/included.json", "r"))
date_threshold = datetime(2021, 4, 1)
vacc_thresh = 0.05
delta_countries = []
for c in countries:
    vacc_data = get_country_vacc_data(c)
    end_time = find_run_end_time(vacc_data, vacc_thresh, c)
    if end_time > date_threshold:
        delta_countries.append(c)

In [ ]:
from emu_renewal.inputs import get_alpha_target, get_cosine_intercept, get_delta_target

In [ ]:
iso3 = "PAN"
iso2 = pycountry.countries.lookup(iso3).alpha_2
continent = pc.country_alpha2_to_continent_code(iso2)
var_data = get_country_vars(iso3)

In [ ]:
vacc_data = get_country_vacc_data(iso3)

In [ ]:
vacc_data.plot()

In [ ]:
end_time = find_run_end_time(vacc_data, 0.05, "BGR")

In [ ]:
end_time

In [ ]:
delta_data = extract_specific_var(var_data, "delta")
max(delta_data.loc[delta_data.index < datetime(2021, 7, 1), "delta_prop"]) > 0.05

In [ ]:
delta_data

In [ ]:
delta_targ = get_delta_target(var_data, continent, end_time)
delta_targ.plot()

In [ ]:
delta_targ

In [ ]:
alpha_targ = get_alpha_target(var_data, continent, end_time)
alpha_targ.plot()

In [ ]:
get_cosine_intercept(alpha_targ, 0)

In [ ]:
BA5_PERIOD_START = datetime(2022, 4, 1)
BA5_PERIOD_END = datetime(2022, 9, 1)
def get_ba5_target(var_data, continent):
    if continent == "OC":
        ba2_data = extract_specific_var(var_data, "ba5")
        period_mask = (BA5_PERIOD_START < ba2_data.index) & (ba2_data.index < BA5_PERIOD_END)
        filt_data = ba2_data[period_mask]
        return filt_data["ba5_prop"]

In [ ]:
get_ba5_target(var_data, continent).plot()

In [ ]:
from emu_renewal.utils import get_row_proportions

In [ ]:
late_data = var_data[(datetime(2022, 4, 1) < var_data.index) & (var_data.index < datetime(2022, 9, 1))]
late_data = late_data[[c for c in late_data.columns if late_data[c].sum() > 0.0]]
get_row_proportions(late_data).plot.area()

In [ ]:

for c in delta_countries[:1]:
    country = pycountry.countries.lookup(c).name
    print(country)
    var_data = get_country_vars(c)
    delta_data = extract_specific_var(var_data, "delta")
    if delta_data is None:
        iso2 = pycountry.countries.lookup(c).alpha_2
        continent = pc.country_alpha2_to_continent_code(iso2)
        cont_data = get_continent_data(continent, "delta")
        delta_data = get_continent_vars(cont_data, "delta")
    filt_data = delta_data[(start < delta_data.index) & (delta_data.index < end)]
    pooled_data = get_dec_pooled_totals(filt_data, "delta")
    plt.figure()
    filt_data["delta_prop"].plot()
    pooled_data["delta_prop"].plot()
    plt.title(country)